# The core layer on other data

`tranpy_core` is not specific to power systems. Two examples: a tabular medical dataset and a synthetic multichannel time series.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from tranpy_core import ExperimentSpec, ModelConfig, SignalDataset, TabularDataset, run_experiment
from tranpy_core.features import WindowStatistics

## Tabular data

In [2]:
raw = load_breast_cancer()
cancer = TabularDataset(features=raw.data, labels=raw.target, feature_names=tuple(raw.feature_names),
                        label_names=tuple(raw.target_names), name="breast_cancer")
spec = ExperimentSpec(name="cancer", models=(ModelConfig("logistic_regression"), ModelConfig("random_forest")), cv_folds=5)
result = run_experiment(spec, cancer)
result.summary_frame()[["accuracy_mean", "roc_auc_mean", "test_accuracy"]].round(4)

,accuracy_mean,roc_auc_mean,test_accuracy
logistic_regression,0.9780,0.9947,0.9825
random_forest,0.9626,0.9922,0.9474


## Time series

Two-channel signals of different lengths; class 1 contains a damped oscillation that grows in one channel. Window statistics turn each signal into a feature vector.

In [3]:
rng = np.random.default_rng(0)
times, values, labels = [], [], []
for i in range(300):
    t = np.linspace(0, rng.uniform(2, 4), rng.integers(80, 160))
    label = i % 2
    growth = 0.6 if label else -0.6
    a = np.exp(growth * t) * np.sin(6 * t) + 0.3 * rng.normal(size=t.size)
    b = np.cos(2 * t) + 0.3 * rng.normal(size=t.size)
    times.append(t); values.append(np.column_stack([a, b])); labels.append(label)
signals = SignalDataset.from_samples(times, values, channel_names=("sensor|a", "sensor|b"), labels=labels,
                                      label_names=("decaying", "growing"), name="oscillations")
signals

SignalDataset('oscillations', samples=300, channels=2, rows=35727, decaying=150, growing=150)

In [4]:
features = WindowStatistics(statistics=("mean", "std", "min", "max", "last"), end=1.0).transform(signals)
spec = ExperimentSpec(name="signals", models=(ModelConfig("logistic_regression"), ModelConfig("gradient_boosting")), cv_folds=5)
run_experiment(spec, features).summary_frame()[["accuracy_mean", "roc_auc_mean", "test_accuracy"]].round(4)

,accuracy_mean,roc_auc_mean,test_accuracy
logistic_regression,1.0,1.0,1.0
gradient_boosting,1.0,1.0,1.0
